<a href="https://colab.research.google.com/github/SHRESHTH121/Demo/blob/main/Aadhaar_Validator_Robust_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install easyocr pyzbar opencv-python-headless --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 15.5 MB/s eta 0:00:00


## Refactored Helper Functions

To make the code less redundant and more organized, I've consolidated various helper functions into logical groups. This includes ensuring tables like `_D_TABLE` and `_P_TABLE` are defined only once, and combining related OCR, image processing, and validation functions. The original scattered function definitions can be ignored or deleted after these new cells are executed.

In [59]:
import cv2
import easyocr
import re
import numpy as np
import os
import json
import tempfile
from google.colab import files

# Initialize EasyOCR reader (can be done once globally)
reader = easyocr.Reader(['en', 'hi'], gpu=False)

In [60]:
# --- Image Preprocessing and Card Frame Detection Functions ---
def preprocess_for_detection(image_path):
    img = cv2.imread(image_path)
    if img is None:
        raise IOError(f"Cannot read image: {image_path}")

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)

    return img, gray, enhanced

def card_fits_frame(
    image_path: str,
    corner_pct: float = 0.08,
    min_corner_std: float = 6.0,
    max_tilt_deg: float = 20.0,
):
    try:
        img, gray, enhanced = preprocess_for_detection(image_path)
    except IOError as e:
        return False, str(e)

    H, W = img.shape[:2]
    ch = max(1, int(H * corner_pct))
    cw = max(1, int(W * corner_pct))

    corner_patches = {
        "top-left":     enhanced[:ch, :cw],
        "top-right":    enhanced[:ch, -cw:],
        "bottom-left":  enhanced[-ch:, :cw],
        "bottom-right": enhanced[-ch:, -cw:],
    }

    corner_stds = {k: float(v.std()) for k, v in corner_patches.items()}

    low_contrast_corners = [k for k, s in corner_stds.items() if s < min_corner_std]
    if len(low_contrast_corners) >= 3:
        return False, (
            f"Card does not fill frame – background visible in: "
            f"{', '.join(low_contrast_corners)} "
            f"(stds={[round(corner_stds[k],1) for k in low_contrast_corners]})"
        )

    scale = min(1.0, 600.0 / max(H, W))
    small_gray = cv2.resize(enhanced, (int(W * scale), int(H * scale)))

    _, otsu = cv2.threshold(small_gray, 0, 255,
                            cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    edges = cv2.Canny(otsu, 50, 150)

    lines = cv2.HoughLinesP(
        edges, 1, np.pi / 180,
        threshold=40,
        minLineLength=int(min(H, W) * scale * 0.10),
        maxLineGap=15,
    )

    tilt = 0.0
    if lines is not None and len(lines) > 0:
        angles = []
        for line in lines:
            x1, y1, x2, y2 = line[0]
            if x2 == x1:
                continue
            angle = np.degrees(np.arctan2(y2 - y1, x2 - x1)) % 180
            if angle > 90:
                angle -= 180
            if angle > 45:
                angle = 90 - angle
            angles.append(abs(angle))
        if angles:
            tilt = float(np.percentile(angles, 25))

    if tilt > max_tilt_deg:
        return False, f"Card tilted by {tilt:.1f}° (limit {max_tilt_deg}°)"

    weakest_std = min(corner_stds.values())
    return True, (
        f"Card fits frame (weakest_corner_std={weakest_std:.1f}, tilt={tilt:.1f}°)"
    )

In [61]:
# --- Blur Detection Functions ---
def calculate_blur_score(image_path):
    img, gray, enhanced = preprocess_for_detection(image_path)
    H, W = gray.shape

    global_score = float(cv2.Laplacian(gray, cv2.CV_64F).var())

    tile_scores = []
    for ri in range(3):
        for ci in range(3):
            tile = gray[ri*H//3:(ri+1)*H//3, ci*W//3:(ci+1)*W//3]
            tile_scores.append(float(cv2.Laplacian(tile, cv2.CV_64F).var()))

    top3_mean = float(np.mean(sorted(tile_scores, reverse=True)[:3]))

    composite = 0.5 * global_score + 0.5 * top3_mean
    return {
        "global":    round(global_score, 2),
        "top3_mean": round(top3_mean, 2),
        "composite": round(composite, 2),
    }


def analyze_aadhaar_blur(
    front_image_path,
    back_image_path,
    front_threshold=35,
    back_threshold=55,
):
    results = {}
    for label, path, threshold in [
        ("front", front_image_path, front_threshold),
        ("back",  back_image_path,  back_threshold),
    ]:
        scores = calculate_blur_score(path)
        is_blurry = scores["composite"] < threshold
        suspicious_sharp = scores["global"] > 2000
        results[label] = {
            "image_path": path,
            **scores,
            "threshold":  threshold,
            "status":     "Blurry" if is_blurry else "Clear",
            "is_blurry":  is_blurry,
            "suspicious_oversharp": suspicious_sharp,
        }
    return results

In [62]:
# --- OCR Extraction Helper Functions ---
def extract_text(image_path, min_conf=0.25):
    results = reader.readtext(image_path, detail=1, paragraph=False)

    lines = []
    for (_, text, conf) in results:
        if conf < min_conf:
            continue
        text = text.strip()
        if re.fullmatch(r'[^A-Za-z0-9\u0900-\u097F]+', text):
            continue
        if not lines or text.lower() != lines[-1].lower():
            lines.append(text)

    return "\n".join(lines)

def extract_aadhaar_number(text):
    ocr_fix = str.maketrans("OIlSB", "01158")
    text_fixed = text.translate(ocr_fix)

    patterns = [
        r'\b(\d{4}[\s\-]\d{4}[\s\-]\d{4})\b',
        r'\b(\d{12})\b',
    ]
    for pat in patterns:
        for match in re.finditer(pat, text_fixed):
            candidate = re.sub(r'[\s\-]', '', match.group(1))
            if len(candidate) == 12 and candidate[0] not in ('0', '1'):
                return match.group(1)
    return None

def extract_dob(text):
    text_fixed = re.sub(r'(?<=[0-9])[Oo](?=[0-9])', '0', text)

    patterns = [
        r'(?:DOB|Date of Birth|D\.O\.B\.?)\s*[:-]?\s*(\d{2}[/.-]\d{2}[/.-]\d{4})',
        r'\b(\d{2}[/.-]\d{2}[/.-]\d{4})\b',
        r'(?:Year of Birth|YOB)\s*[:-]?\s*(\d{4})',
    ]
    for pat in patterns:
        m = re.search(pat, text_fixed, re.IGNORECASE)
        if m:
            return m.group(1)
    return None

def extract_gender(text):
    text_lower = text.lower().replace(' ', '')

    gender_map = [
        (r'female|femal|महिला', 'Female'),
        (r'male|purus|पुरुष',   'Male'),
        (r'transgender|तृतीय', 'Transgender'),
    ]
    for pattern, label in gender_map:
        if re.search(pattern, text_lower):
            return label
    return None

def extract_name(text):
    lines = [l.strip() for l in text.split('\n') if l.strip()]

    NOISE = {
        'government', 'india', 'dob', 'male', 'female', 'transgender',
        'aadhaar', 'address', 'uid', 'uidai', 'enrolment', 'vid',
        'unique', 'identification', 'authority', 'help', 'www', 'of',
        'download', 'digitally', 'signed', 'verified',
    }

    def is_valid_name_line(line):
        if len(line.split()) < 2:
            return False
        if re.search(r'\d', line):
            return False
        if any(w in line.lower() for w in NOISE):
            return False
        alpha_ratio = sum(c.isalpha() for c in line) / max(len(line), 1)
        if alpha_ratio < 0.70:
            return False
        return True

    for i, line in enumerate(lines):
        if re.search(r'government\s+of\s+india', line, re.IGNORECASE):
            for candidate in lines[i+1 : i+4]:
                if is_valid_name_line(candidate):
                    return candidate
            break

    for line in lines:
        if is_valid_name_line(line):
            return line

    return None

def extract_address(text):
    text = re.sub(
        r'Details\s+as\s+on\s*[:-]?\s*[\d/]+',
        '',
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(r'\b([SDWC]/O\s*:)', r'\n\1', text, flags=re.IGNORECASE)
    text = re.sub(r'\bAddress\s*:', r'\nAddress:\n', text, flags=re.IGNORECASE)

    lines = [l.strip() for l in text.split('\n') if l.strip()]

    START_PATTERNS = [
        r'\baddr(ess)?\b',
        r'\bc/o\b', r'\bs/o\b', r'\bd/o\b', r'\bw/o\b',
        r'\bhouse\b', r'\bflat\b', r'\bplot\b', r'\bvill(age)?\b',
        r'#\s*\d',
        r'\bward\b', r'\bnear\b', r'\bsector\b',
    ]
    STOP_PATTERNS = [
        r'\bvid\b', r'uidai', r'www\.', r'\b1947\b',
        r'help\s*centre', r'toll[\s-]?free', r'download',
        r'unique\s+id', r'help@',
    ]

    collecting = False
    address_lines = []

    for line in lines:
        line_l = line.lower()

        if collecting and any(re.search(p, line_l) for p in STOP_PATTERNS):
            break
        if collecting and len(address_lines) >= 10:
            break

        if not collecting and any(re.search(p, line_l) for p in START_PATTERNS):
            collecting = True
            if re.fullmatch(r'addr(ess)?\s*:?', line_l):
                continue

        if collecting:
            if re.search(r'\b\d{4}\s?\d{4}\s?\d{4}\b', line):
                continue
            address_lines.append(line)

    if not address_lines:
        full = ' '.join(lines)
        m = re.search(r'(.{0,200})\b(\d{6})\b', full)
        if m:
            return m.group(0).strip()

    return '\n'.join(address_lines).strip()

def get_lines_easyocr(image_path, min_conf=0.20):
    results = reader.readtext(image_path, detail=1, paragraph=False)
    lines = []
    for (bbox, text, conf) in results:
        if conf >= min_conf:
            lines.append({'text': text.strip(), 'confidence': conf})
    return lines

def lines_to_text(lines):
    return '\n'.join([line['text'] for line in lines])

def extract_aadhaar_information(front_image, back_image):
    front_text = extract_text(front_image)
    back_text  = extract_text(back_image)

    aadhaar_data = {
        "front_text":     front_text,
        "back_text":      back_text,
        "name":           extract_name(front_text),
        "dob":            extract_dob(front_text),
        "gender":         extract_gender(front_text),
        "aadhaar_number": extract_aadhaar_number(front_text),
        "address":        extract_address(back_text),
        "photo_present":  detect_photo(front_image),
        "qr_present":     detect_qr_presence(back_image), # Placeholder, will be replaced by actual QR detection result
    }
    return aadhaar_data

def validate_completeness(aadhaar_data):
    mandatory = ["name", "dob", "gender", "aadhaar_number", "address"]
    missing = []
    for field in mandatory:
        val = aadhaar_data.get(field)
        if not val or (isinstance(val, str) and not val.strip()):
            missing.append(field)
    return missing

In [63]:
# --- Photo and QR Code Detection Functions ---
def detect_photo(image_path):
    img, gray, enhanced = preprocess_for_detection(image_path)

    cascade_paths = [
        cv2.data.haarcascades + 'haarcascade_frontalface_default.xml',
        cv2.data.haarcascades + 'haarcascade_frontalface_alt2.xml',
        cv2.data.haarcascades + 'haarcascade_profileface.xml',
    ]

    for cp in cascade_paths:
        det = cv2.CascadeClassifier(cp)
        if det.empty():
            continue
        for image_to_try in [gray, enhanced]:
            faces = det.detectMultiScale(
                image_to_try,
                scaleFactor=1.05,
                minNeighbors=3,
                minSize=(30, 30),
                flags=cv2.CASCADE_SCALE_IMAGE,
            )
            if len(faces) > 0:
                return True

    return False

def _sharpen_for_qr(gray):
    blur = cv2.GaussianBlur(gray, (0, 0), 2.0)
    sharp = cv2.addWeighted(gray, 1.8, blur, -0.8, 0)
    return sharp


def _try_decode_variants(detector, image):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(image)
    sharpened = _sharpen_for_qr(image)
    binarized = cv2.adaptiveThreshold(
        image, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 11, 2,
    )
    for variant_name, variant in [
        ("raw", image),
        ("enhanced", enhanced),
        ("sharpened", sharpened),
        ("binarized", binarized),
    ]:
        data, bbox, _ = detector.detectAndDecode(variant)
        if bbox is not None and data:
            return data, variant_name
    return None, None


def detect_qr_presence(image_path):
    try:
        img, gray, enhanced = preprocess_for_detection(image_path)
    except IOError as e:
        return {"qr_present": False, "reason": str(e)}

    H, W = img.shape[:2]

    qr_det = cv2.QRCodeDetector()
    data, variant = _try_decode_variants(qr_det, gray)
    if data:
        return {"qr_present": True, "method": f"cv2.QRCodeDetector({variant})",
                "data_length": len(data)}

    quadrants = {
        "TL": gray[:H//2, :W//2], "TR": gray[:H//2, W//2:],
        "BL": gray[H//2:, :W//2], "BR": gray[H//2:, W//2:],
    }
    for qname, quad in quadrants.items():
        quad_up = cv2.resize(quad, None, fx=1.5, fy=1.5, interpolation=cv2.INTER_CUBIC)
        data, variant = _try_decode_variants(qr_det, quad_up)
        if data:
            return {"qr_present": True,
                    "method": f"cv2.QRCodeDetector({qname},{variant})",
                    "data_length": len(data)}

    try:
        wechat_det = cv2.wechat_qrcode_WeChatQRCode()
        for image_to_try in [gray, enhanced, _sharpen_for_qr(gray)]:
            texts, points = wechat_det.detectAndDecode(image_to_try)
            if texts:
                return {"qr_present": True, "method": "WeChatQR",
                        "data_length": len(texts[0])}
    except (cv2.error, AttributeError):
        pass

    try:
        from pyzbar.pyzbar import decode as pyz_decode
        for image_to_try in [img, cv2.cvtColor(_sharpen_for_qr(gray), cv2.COLOR_GRAY2BGR)]:
            decoded = pyz_decode(image_to_try)
            if decoded:
                sym = decoded[0]
                return {"qr_present": True, "method": f"pyzbar({sym.type})",
                        "data_length": len(sym.data)}
    except ImportError:
        pass

    blurred = cv2.GaussianBlur(enhanced, (5, 5), 0)
    _, bw = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    closed = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel, iterations=2)

    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    image_area = H * W

    corner_regions = [
        ("TR", gray[: H//3, W//2 :]),
        ("BR", gray[H//2 :, W//2 :]),
        ("BL", gray[H//2 :, : W//2]),
    ]
    for cname, region in corner_regions:
        if region.size == 0:
            continue
        rH, rW = region.shape
        sharp_r = _sharpen_for_qr(region)
        _, rbw = cv2.threshold(sharp_r, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        density = float(rbw.mean()) / 255.0
        if 0.25 <= density <= 0.68 and float(cv2.Laplacian(region, cv2.CV_64F).var()) > 50:
            return {
                "qr_present": True,
                "method":     f"morphological_corner({cname})",
                "density":    round(density, 3),
                "lap_var":    round(float(cv2.Laplacian(region, cv2.CV_64F).var()), 1),
            }

    for cnt in sorted(contours, key=cv2.contourArea, reverse=True)[:25]:
        area = cv2.contourArea(cnt)
        if area < 0.008 * image_area or area > 0.40 * image_area:
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        aspect = w / float(h)
        if not (0.72 <= aspect <= 1.38):
            continue
        roi = bw[y:y+h, x:x+w]
        density = roi.mean() / 255.0
        if 0.22 <= density <= 0.72:
            return {
                "qr_present": True,
                "method":     "morphological_global",
                "bbox":       (x, y, w, h),
                "density":    round(density, 3),
            }

    return {"qr_present": False, "reason": "No QR code detected by any method"}

def detect_qr_box_presence(image_path):
    # This function is now just an alias, avoiding redundancy
    return detect_qr_presence(image_path)

In [64]:
# --- Aadhaar Number Validation (Verhoeff Algorithm) Functions ---
_D_TABLE = [
    [0,1,2,3,4,5,6,7,8,9],
    [1,2,3,4,0,6,7,8,9,5],
    [2,3,4,0,1,7,8,9,5,6],
    [3,4,0,1,2,8,9,5,6,7],
    [4,0,1,2,3,9,5,6,7,8],
    [5,9,8,7,6,0,4,3,2,1],
    [6,5,9,8,7,1,0,4,3,2],
    [7,6,5,9,8,2,1,0,4,3],
    [8,7,6,5,9,3,2,1,0,4],
    [9,8,7,6,5,4,3,2,1,0],
]
_P_TABLE = [
    [0,1,2,3,4,5,6,7,8,9],
    [1,5,7,6,2,8,3,0,9,4],
    [5,8,0,3,7,9,6,1,4,2],
    [8,9,1,6,0,4,3,5,2,7],
    [9,4,5,3,1,2,6,8,7,0],
    [4,2,8,6,5,7,3,9,0,1],
    [2,7,9,3,8,0,6,4,1,5],
    [7,0,4,6,9,1,3,2,5,8],
]

def validate_aadhaar_number(aadhaar_number):
    if aadhaar_number is None:
        return False

    ocr_fix = str.maketrans("OIlSB", "01158")
    cleaned = re.sub(r'[\s\-]', '', str(aadhaar_number).translate(ocr_fix))

    if not cleaned.isdigit():
        return False
    if len(cleaned) != 12:
        return False
    if cleaned[0] in ('0', '1'):
        return False

    c = 0
    for i, digit in enumerate(reversed(cleaned)):
        c = _D_TABLE[c][_P_TABLE[i % 8][int(digit)]]
    return c == 0

def aadhaar_validation_report(aadhaar_data):
    number   = aadhaar_data.get("aadhaar_number")
    is_valid = validate_aadhaar_number(number)
    return {
        "aadhaar_number":  number,
        "checksum_valid":  is_valid,
        "fraud_signal":    not is_valid,
    }

In [65]:
# --- Document Validation Function ---
def validate_aadhaar_document(
        image_path,
        min_confidence=50):

    score = 0
    reasons = []

    # ---------------------------------
    # Face Detection
    # ---------------------------------
    try:
        photo_present = detect_photo(image_path)
        if photo_present:
            score += 20
        else:
            reasons.append("Photo not detected")
    except Exception as e:
        reasons.append(f"Photo detection failed: {str(e)}")

    # ---------------------------------
    # QR Detection
    # ---------------------------------
    try:
        qr_result = detect_qr_box_presence(image_path)
        qr_present = qr_result.get("qr_present", False)

        if qr_present:
            score += 20
        else:
            reasons.append("QR not detected")
    except Exception as e:
        reasons.append(f"QR detection failed: {str(e)}")

    # ---------------------------------
    # OCR Classification
    # ---------------------------------
    try:
        lines = get_lines_easyocr(image_path, min_conf=0.20)
        text = lines_to_text(lines)
        text_lower = text.lower()

        # -----------------------------
        # Aadhaar Keywords
        # -----------------------------
        keyword_weights = {
            "government of india": 10,
            "aadhaar": 10,
            "uidai": 10,
            "year of birth": 5,
            "dob": 5,
            "male": 5,
            "female": 5,
            "address": 5,
        }

        keyword_score = 0
        matched_keywords = []
        for keyword, weight in keyword_weights.items():
            if keyword in text_lower:
                keyword_score += weight
                matched_keywords.append(keyword)

        score += min(keyword_score, 30)

        # -----------------------------
        # Aadhaar Number
        # -----------------------------
        aadhaar_number = extract_aadhaar_number(text)
        if aadhaar_number:
            score += 15
            if validate_aadhaar_number(aadhaar_number):
                score += 15
            else:
                reasons.append("Aadhaar checksum invalid")
        else:
            reasons.append("Aadhaar number not found")

    except Exception as e:
        reasons.append(f"OCR classification failed: {str(e)}")

    # ---------------------------------
    # Final Decision
    # ---------------------------------
    score = min(score, 100)
    is_aadhaar = (score >= min_confidence)

    return {
        "is_aadhaar": is_aadhaar,
        "confidence_score": score,
        "reasons": reasons,
        "threshold": min_confidence
    }

In [66]:
# --- Image Tampering Detection Function ---
def detect_image_tampering(image_path):

    img = cv2.imread(image_path)

    if img is None:
        return {
            "edited": False,
            "confidence": 0,
            "reason": "Unable to read image"
        }

    score = 0
    reasons = []

    # =====================================
    # 1. Error Level Analysis (ELA)
    # =====================================

    tmp_file = tempfile.NamedTemporaryFile(
        suffix=".jpg",
        delete=False
    )

    temp_path = tmp_file.name

    cv2.imwrite(
        temp_path,
        img,
        [cv2.IMWRITE_JPEG_QUALITY, 90]
    )

    recompressed = cv2.imread(temp_path)

    os.remove(temp_path)

    ela = cv2.absdiff(
        img,
        recompressed
    )

    ela_score = float(
        np.mean(ela)
    )

    if ela_score > 8:
        score += 30
        reasons.append(
            f"High ELA score ({ela_score:.2f})"
        )

    # =====================================
    # 2. Sharpness Inconsistency
    # =====================================

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    h, w = gray.shape

    tile_scores = []

    for r in range(3):
        for c in range(3):

            tile = gray[
                r*h//3:(r+1)*h//3,
                c*w//3:(c+1)*w//3
            ]

            lap = cv2.Laplacian(
                tile,
                cv2.CV_64F
            ).var()

            tile_scores.append(lap)

    sharp_std = np.std(tile_scores)

    if sharp_std > 250:
        score += 20
        reasons.append(
            f"Sharpness inconsistency ({sharp_std:.1f})"
        )

    # =====================================
    # 3. Noise Inconsistency
    # =====================================

    noise_levels = []

    for r in range(3):
        for c in range(3):

            tile = gray[
                r*h//3:(r+1)*h//3,
                c*w//3:(c+1)*w//3
            ]

            blur = cv2.GaussianBlur(
                tile,
                (3,3),
                0
            )

            noise = np.std(
                tile.astype(np.float32)
                -
                blur.astype(np.float32)
            )

            noise_levels.append(noise)

    noise_std = np.std(noise_levels)

    if noise_std > 8:
        score += 20
        reasons.append(
            f"Noise inconsistency ({noise_std:.2f})"
        )

    # =====================================
    # 4. Edge Density Inconsistency
    # =====================================

    densities = []

    for r in range(3):
        for c in range(3):

            tile = gray[
                r*h//3:(r+1)*h//3,
                c*w//3:(c+1)*w//3
            ]

            edges = cv2.Canny(
                tile,
                50,
                150
            )

            density = np.mean(
                edges > 0
            )

            densities.append(density)

    density_std = np.std(densities)

    if density_std > 0.10:
        score += 15
        reasons.append(
            f"Edge inconsistency ({density_std:.3f})"
        )

    # =====================================
    # Final Decision
    # =====================================

    score = min(score, 100)

    edited = score >= 40

    return {

        "edited": edited,

        "tampering_score": score,

        "reasons": reasons,

        "ela_score": round(ela_score, 2),

        "sharpness_std": round(sharp_std, 2),

        "noise_std": round(noise_std, 2),

        "edge_density_std": round(density_std, 3)
    }

In [67]:
# --- Address Processing Functions ---
def clean_address(raw_address):
    if not raw_address:
        return None
    addr = raw_address.replace('\n', ' ')
    addr = re.sub(r'\s+', ' ', addr)
    addr = re.sub(r'^(addr(ess)?\s*[:-]?\s*)', '', addr, flags=re.IGNORECASE)
    return addr.strip()


def extract_guardian_details(address):
    if not address:
        return {"guardian_type": None, "guardian_name": None}

    patterns = [
        (r'S[\s/]*O[\s:]+([A-Za-z][A-Za-z .]+)',  'S/O'),
        (r'D[\s/]*O[\s:]+([A-Za-z][A-Za-z .]+)',  'D/O'),
        (r'W[\s/]*O[\s:]+([A-Za-z][A-Za-z .]+)',  'W/O'),
        (r'C[\s/]*O[\s:]+([A-Za-z][A-Za-z .]+)',  'C/O'),
        (r'Son\s+of[\s:]+([A-Za-z][A-Za-z .]+)',  'S/O'),
        (r'Daughter\s+of[\s:]+([A-Za-z][A-Za-z .]+)', 'D/O'),
        (r'Wife\s+of[\s:]+([A-Za-z][A-Za-z .]+)',     'W/O'),
    ]

    for pattern, gtype in patterns:
        m = re.search(pattern, address, re.IGNORECASE)
        if m:
            name = m.group(1).strip()
            name = re.split(r'[\d,]', name)[0].strip()
            noise = r'\b(house|flat|plot|village|vill|po|ps|dist|pin)\b'
            name = re.split(noise, name, flags=re.IGNORECASE)[0].strip()
            return {"guardian_type": gtype, "guardian_name": name}

    return {"guardian_type": None, "guardian_name": None}


def extract_pincode(address):
    if not address:
        return None
    for m in re.finditer(r'\b(\d{6})\b', address):
        candidate = m.group(1)
        if not re.match(r'(19|20)\d{4}', candidate):
            return candidate
    return None


def extract_city_state(address):
    if not address:
        return None, None
    parts = [p.strip() for p in re.split(r'[\n,]', address)]
    pincode = extract_pincode(address)
    city, state = None, None
    for i, part in enumerate(parts):
        if pincode and pincode in part:
            if i >= 1:
                city  = parts[i-1]
            if i >= 2:
                state = parts[i-2]
            break
    return city, state


def process_address(raw_address):
    cleaned  = clean_address(raw_address)
    guardian = extract_guardian_details(cleaned)
    pincode  = extract_pincode(cleaned)
    city, state = extract_city_state(cleaned)

    return {
        "cleaned_address": cleaned,
        "guardian_type":   guardian["guardian_type"],
        "guardian_name":   guardian["guardian_name"],
        "city":            city,
        "state":           state,
        "pincode":         pincode,
    }

In [68]:
# --- Final Verification Decision Function ---
def final_verification_decision(
        merged,
        blur_report,
        front_doc_check,
        back_doc_check,
        front_tamper,
        back_tamper,
        missing_fields,
        needs_review):

    score = 100
    penalties = []

    # ==================================
    # DOCUMENT DETECTION
    # ==================================
    best_doc_conf = max(
        front_doc_check["confidence_score"],
        back_doc_check["confidence_score"]
    )

    if best_doc_conf < 50:
        score -= 40
        penalties.append(
            f"Low Aadhaar confidence ({best_doc_conf})"
        )
    elif best_doc_conf < 70:
        score -= 15
        penalties.append(
            f"Moderate Aadhaar confidence ({best_doc_conf})"
        )

    # ==================================
    # TAMPERING
    # ==================================
    worst_tamper = max(
        front_tamper["tampering_score"],
        back_tamper["tampering_score"]
    )

    if worst_tamper >= 70:
        score -= 50
        penalties.append(
            f"High tampering risk ({worst_tamper})"
        )
    elif worst_tamper >= 40:
        score -= 20
        penalties.append(
            f"Medium tampering risk ({worst_tamper})"
        )

    # ==================================
    # BLUR
    # ==================================
    if blur_report["front"]["is_blurry"]:
        score -= 15
        penalties.append("Front image blurry")
    if blur_report["back"]["is_blurry"]:
        score -= 15
        penalties.append("Back image blurry")

    # ==================================
    # PHOTO CHECK
    # ==================================
    if not merged.get("photo_present", False):
        score -= 20
        penalties.append("Photo not detected")

    # ==================================
    # QR CHECK
    # ==================================
    if not merged.get("qr_present", {}).get("qr_present", False): # Access qr_present within the nested dict
        score -= 20
        penalties.append("QR not detected")

    # ==================================
    # AADHAAR NUMBER
    # ==================================
    if not merged.get("aadhaar_number"):
        score -= 40
        penalties.append("Aadhaar number missing")
    elif not merged.get("aadhaar_checksum_valid", False):
        score -= 30
        penalties.append("Invalid Aadhaar checksum")

    # ==================================
    # MANDATORY FIELDS
    # ==================================
    if missing_fields:
        penalty = min(len(missing_fields) * 10, 40)
        score -= penalty
        penalties.append(f"Missing fields: {missing_fields}")

    # ==================================
    # OCR DISAGREEMENTS
    # ==================================
    if needs_review:
        score -= min(len(needs_review) * 5, 20)
        penalties.append(f"OCR disagreements: {needs_review}")

    # ==================================
    # CLAMP
    # ==================================
    score = max(min(score, 100), 0)

    # ==================================
    # FINAL DECISION
    # ==================================
    if score >= 80:
        decision = "PASS"
    elif score >= 50:
        decision = "REVIEW"
    else:
        decision = "FAIL"

    return {
        "decision": decision,
        "confidence_score": score,
        "penalties": penalties,
        "document_confidence": best_doc_conf,
        "tampering_score": worst_tamper
    }

In [69]:
# Removed redundant import statements, as they are now handled in an earlier consolidated cell.

In [49]:
print("Upload Front Aadhaar Image")
front_upload = files.upload()

print("Upload Back Aadhaar Image")
back_upload = files.upload()

front_image_path = next(iter(front_upload))
back_image_path  = next(iter(back_upload))

print("Front:", front_image_path)
print("Back :", back_image_path)

Upload Front Aadhaar Image


Saving adhaar front side edited.jpeg to adhaar front side edited.jpeg
Upload Back Aadhaar Image


Saving adhaar backside.jpeg to adhaar backside (1).jpeg
Front: adhaar front side edited.jpeg
Back : adhaar backside (1).jpeg


In [70]:
# Removed redundant EasyOCR reader initialization, as it is now handled in an earlier consolidated cell.

# **Document Validation**

In [71]:
# Removed redundant 'preprocess_for_detection' and 'card_fits_frame' function definitions, as they are now handled in an earlier consolidated cell.

In [50]:
for label, path in [("Front", front_image_path), ("Back", back_image_path)]:
    ok, msg = card_fits_frame(path)
    print(f"{label}: {'PASS ✓' if ok else 'FAIL ✗'}  —  {msg}")

Front: PASS ✓  —  Card fits frame (weakest_corner_std=46.0, tilt=0.0°)
Back: PASS ✓  —  Card fits frame (weakest_corner_std=51.1, tilt=1.2°)


# **Blur Detection**

In [72]:
# Removed redundant blur detection function definitions ('calculate_blur_score', 'analyze_aadhaar_blur'), as they are now handled in an earlier consolidated cell.

In [51]:
blur_report = analyze_aadhaar_blur(
    front_image_path, back_image_path,
    front_threshold=35,
    back_threshold=55,
)
print("\n========== BLUR REPORT ==========\n")
for side, info in blur_report.items():
    status = info['status']
    print(f"{side.upper()}: {status}  "
          f"(composite={info['composite']}, global={info['global']}, "
          f"top3_mean={info['top3_mean']})")
    if info.get('suspicious_oversharp'):
        print(f"  ⚠ Suspiciously high sharpness — may be a scan or photocopy")


========== BLUR REPORT ==========

FRONT: Clear  (composite=132.57, global=89.68, top3_mean=175.46)
BACK: Clear  (composite=165.89, global=123.43, top3_mean=208.36)


# **OCR Extraction and Image verification**
to check if the uploaded image is of adhaar card or something else

In [73]:
# Removed redundant OCR extraction function definition ('extract_text'), as it is now handled in an earlier consolidated cell.

In [74]:
# Removed redundant OCR extraction function definition ('extract_aadhaar_number'), as it is now handled in an earlier consolidated cell.

In [75]:
# Removed redundant OCR extraction function definition ('extract_dob'), as it is now handled in an earlier consolidated cell.

In [76]:
# Removed redundant OCR extraction function definition ('extract_gender'), as it is now handled in an earlier consolidated cell.

In [77]:
# Removed redundant OCR extraction function definition ('extract_name'), as it is now handled in an earlier consolidated cell.

In [78]:
# Removed redundant OCR extraction function definition ('extract_address'), as it is now handled in an earlier consolidated cell.

In [79]:
# Removed redundant photo detection function definition ('detect_photo'), as it is now handled in an earlier consolidated cell.

In [80]:
# Removed redundant QR code detection functions ('_sharpen_for_qr', '_try_decode_variants', 'detect_qr_presence'), as they are now handled in an earlier consolidated cell.

In [81]:
# Removed redundant QR box presence detection function ('detect_qr_box_presence'), as it is now handled in an earlier consolidated cell.

In [82]:
# Removed redundant EasyOCR line retrieval function ('get_lines_easyocr'), as it is now handled in an earlier consolidated cell.

In [83]:
# Removed redundant 'lines_to_text' and 'validate_aadhaar_number' function definitions, as they are now handled in earlier consolidated cells.

In [84]:
# Removed redundant 'extract_aadhaar_information' function definition, as it is now handled in an earlier consolidated cell.

In [85]:
# Removed redundant 'validate_completeness' function definition, as it is now handled in an earlier consolidated cell.

In [86]:
# Removed redundant 'validate_aadhaar_document' function definition, as it is now handled in an earlier consolidated cell.

In [52]:
front_doc_check = (
    validate_aadhaar_document(
        front_image_path
    )
)

back_doc_check = (
    validate_aadhaar_document(
        back_image_path
    )
)

print(
    "\n===== DOCUMENT VALIDATION ====="
)

print(
    "Front:",
    front_doc_check
)

print(
    "Back:",
    back_doc_check
)

if not (

    front_doc_check[
        "is_aadhaar"
    ]

    or

    back_doc_check[
        "is_aadhaar"
    ]
):

    raise ValueError(

        f"""
        Uploaded document does not appear to be an Aadhaar card.

        Front Confidence:
        {front_doc_check['confidence_score']}

        Back Confidence:
        {back_doc_check['confidence_score']}

        Front Reasons:
        {front_doc_check['reasons']}

        Back Reasons:
        {back_doc_check['reasons']}
        """
    )

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



===== DOCUMENT VALIDATION =====
Front: {'is_aadhaar': True, 'confidence_score': 75, 'reasons': [], 'threshold': 50}
Back: {'is_aadhaar': True, 'confidence_score': 70, 'reasons': [], 'threshold': 50}


In [53]:
aadhaar_data = extract_aadhaar_information(front_image_path, back_image_path)
missing_fields = validate_completeness(aadhaar_data)

print("\n========== EXTRACTED DATA ==========\n")
for k, v in aadhaar_data.items():
    if k not in ('front_text', 'back_text'):
        print(f"  {k}: {v}")

print("\nMissing fields:", missing_fields if missing_fields else "None ✓")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



========== EXTRACTED DATA ==========

  name: Shreshth Garg
  dob: 13/07/2005
  gender: Male
  aadhaar_number: 7445 3447 0565
  address: Acaress: SO:Padcep Kumar = ५३५/२. randhawa voad nard no  ७ Kharar PO: Kharar DIST: SAS Nagar Mohali). Puniab 140301
  photo_present: True
  qr_present: {'qr_present': True, 'method': 'morphological_corner(TR)', 'density': 0.352, 'lap_var': 111.3}

Missing fields: None ✓


# **Aadhaar Number Validation (Verhoeff)**

In [87]:
# Removed redundant Aadhaar number validation related functions and tables ('_D_TABLE', '_P_TABLE', 'validate_aadhaar_number', 'aadhaar_validation_report'), as they are now handled in an earlier consolidated cell.

In [54]:
validation_report = aadhaar_validation_report(aadhaar_data)
aadhaar_data["aadhaar_checksum_valid"] = validation_report["checksum_valid"]
print("Validation report:", validation_report)

Validation report: {'aadhaar_number': '7445 3447 0565', 'checksum_valid': True, 'fraud_signal': False}


# **Address Parsing**

In [88]:
# Removed redundant address processing functions ('clean_address', 'extract_guardian_details', 'extract_pincode', 'extract_city_state', 'process_address'), as they are now handled in an earlier consolidated cell.

In [55]:
address_info = process_address(aadhaar_data["address"])
print("Address breakdown:", address_info)

aadhaar_data.update(address_info)
del aadhaar_data["address"]

Address breakdown: {'cleaned_address': 'Acaress: SO:Padcep Kumar = ५३५/२. randhawa voad nard no ७ Kharar PO: Kharar DIST: SAS Nagar Mohali). Puniab 140301', 'guardian_type': 'S/O', 'guardian_name': 'Padcep Kumar', 'city': None, 'state': None, 'pincode': '140301'}


In [56]:
print("\n========== FINAL AADHAAR DATA ==========\n")
print(json.dumps(
    {k: v for k, v in aadhaar_data.items() if k not in ('front_text', 'back_text')},
    indent=2, default=str
))


========== FINAL AADHAAR DATA ==========

{
  "name": "Shreshth Garg",
  "dob": "13/07/2005",
  "gender": "Male",
  "aadhaar_number": "7445 3447 0565",
  "photo_present": true,
  "qr_present": {
    "qr_present": true,
    "method": "morphological_corner(TR)",
    "density": 0.352,
    "lap_var": 111.3
  },
  "aadhaar_checksum_valid": true,
  "cleaned_address": "Acaress: SO:Padcep Kumar = \u096b\u0969\u096b/\u0968. randhawa voad nard no \u096d Kharar PO: Kharar DIST: SAS Nagar Mohali). Puniab 140301",
  "guardian_type": "S/O",
  "guardian_name": "Padcep Kumar",
  "city": null,
  "state": null,
  "pincode": "140301"
}


In [ ]:
def extract_aadhaar_information(front_image, back_image):
    front_text = extract_text(front_image)
    back_text  = extract_text(back_image)

    aadhaar_data = {
        "front_text":     front_text,
        "back_text":      back_text,
        "name":           extract_name(front_text),
        "dob":            extract_dob(front_text),
        "gender":         extract_gender(front_text),
        "aadhaar_number": extract_aadhaar_number(front_text),
        "address":        extract_address(back_text),
        "photo_present":  detect_photo(front_image),
        "qr_present":     detect_qr_presence(back_image),
    }
    return aadhaar_data

In [89]:
# Removed redundant 'detect_image_tampering' function definition, as it is now handled in an earlier consolidated cell.

In [57]:
# =====================================
# Tampering Validation Gate
# =====================================

front_tamper = detect_image_tampering(front_image_path)
back_tamper  = detect_image_tampering(back_image_path)

print("\n===== TAMPERING ANALYSIS =====")
print("Front:", front_tamper)
print("Back :", back_tamper)

TAMPER_THRESHOLD = 70

if front_tamper["tampering_score"] >= TAMPER_THRESHOLD:

    raise ValueError(
        f"""
        HIGH RISK OF IMAGE TAMPERING DETECTED (FRONT)

        Tampering Score:
        {front_tamper['tampering_score']}

        Reasons:
        {front_tamper['reasons']}

        Please upload the original Aadhaar image.
        """
    )

if back_tamper["tampering_score"] >= TAMPER_THRESHOLD:

    raise ValueError(
        f"""
        HIGH RISK OF IMAGE TAMPERING DETECTED (BACK)

        Tampering Score:
        {back_tamper['tampering_score']}

        Reasons:
        {back_tamper['reasons']}

        Please upload the original Aadhaar image.
        """
    )


===== TAMPERING ANALYSIS =====
Front: {'edited': False, 'tampering_score': 0, 'reasons': [], 'ela_score': 0.09, 'sharpness_std': np.float64(66.92), 'noise_std': np.float32(0.79), 'edge_density_std': np.float64(0.026)}
Back : {'edited': False, 'tampering_score': 0, 'reasons': [], 'ela_score': 0.31, 'sharpness_std': np.float64(81.16), 'noise_std': np.float32(0.74), 'edge_density_std': np.float64(0.047)}


In [90]:
# Removed redundant 'final_verification_decision' function definition, as it is now handled in an earlier consolidated cell.

In [48]:
needs_review = []
verification_result = final_verification_decision(
    merged=aadhaar_data,
    blur_report=blur_report,
    front_doc_check=front_doc_check,
    back_doc_check=back_doc_check,
    front_tamper=front_tamper,
    back_tamper=back_tamper,
    missing_fields=missing_fields,
    needs_review=needs_review
)

print("\n===== FINAL DECISION =====")

print(
    json.dumps(
        verification_result,
        indent=2
    )
)


===== FINAL DECISION =====
{
  "decision": "PASS",
  "confidence_score": 100,
  "penalties": [],
  "document_confidence": 95,
  "tampering_score": 0
}
